# Fooocus no Google Colab — um play e pronto

Gera **fotos** por inteligência artificial numa página com botões. É o painel de imagem
mais fácil que existe: você escreve o que quer e clica em Generate.

### Antes de apertar o play

Menu do topo → **Ambiente de execução** → **Alterar o tipo de ambiente de execução** →
marque **T4 GPU** → **Salvar**. Sem isso nada funciona.

### Depois

Aperte o play, autorize o Google Drive quando ele pedir, e **espere o link azul terminado
em `.gradio.live`**. Esse link é o painel.

**Não feche esta aba** enquanto estiver usando o painel.


In [ ]:
#@title ## Fooocus no Colab — aperte o play e espere o link azul { display-mode: "form" }
#@markdown Painel de **imagem** (fotos). Nao precisa mexer em nada aqui embaixo.
USAR_GOOGLE_DRIVE = True  #@param {type:"boolean"}
#@markdown Pasta dentro do seu Drive onde ficam as fotos geradas e as LoRAs:
PASTA_NO_DRIVE = "Fooocus"  #@param {type:"string"}

import os, json, subprocess

# ------------------------------------------------------------------ 1. a placa
gpu = ""
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
except Exception:
    pass

if not gpu:
    print()
    print("!" * 76)
    print("!!  PARE. ESTA SESSAO ESTA SEM PLACA DE VIDEO - E SEM ELA O PAINEL NAO LIGA.")
    print("!!")
    print("!!  Sao DUAS causas possiveis. Veja qual foi a sua:")
    print("!!")
    print("!!  1) VOCE AINDA NAO ESCOLHEU A PLACA.")
    print("!!     Menu do topo  ->  Ambiente de execucao  ->  Alterar o tipo de")
    print("!!     ambiente de execucao  ->  marque  T4 GPU  ->  Salvar.")
    print("!!     Depois aperte o play aqui de novo.")
    print("!!")
    print("!!  2) O COLAB AVISOU  -nao e possivel conectar a GPU / limites de uso-")
    print("!!     e voce clicou em  -Conectar sem GPU-.")
    print("!!     Entao a sua cota gratuita de placa de video acabou por hoje.")
    print("!!     Nao ha nada para ajustar aqui: quem fechou a torneira foi o Google.")
    print("!!     Suas saidas sao tres:")
    print("!!       . esperar (costuma voltar em algumas horas, ou no dia seguinte)")
    print("!!       . abrir este mesmo link com a OUTRA conta do Google")
    print("!!       . usar o Fooocus do seu proprio computador:")
    print("!!         Documentos > Fooocus_win64_2-1-831 > run.bat")
    print("!" * 76)
    print()
    raise RuntimeError("Sem placa de video. Leia o aviso acima: ou falta escolher T4 GPU, ou a cota diaria de GPU do Colab acabou por hoje.")

print("Placa de video: " + gpu)

# ------------------------------------------------------------------ 2. o Drive
PASTA = None
if USAR_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA = '/content/drive/MyDrive/' + PASTA_NO_DRIVE.strip('/ ')
    for sub in ('loras', 'saida'):
        os.makedirs(os.path.join(PASTA, sub), exist_ok=True)
    print("Drive conectado. Fotos e LoRAs em: " + PASTA)
else:
    print("Sem Drive: as fotos somem quando o Colab fechar. Baixe antes de sair.")

# ------------------------------------------------------------------ 3. o programa
!pip install -q pygit2==1.15.1
%cd /content
if not os.path.isdir('/content/Fooocus'):
    print("\nBaixando o programa Fooocus (uma vez so)...")
    !git clone -q https://github.com/lllyasviel/Fooocus.git
%cd /content/Fooocus
!git -C /content/Fooocus pull -q

# ------------------------------------------------------------------ 4. mandar salvar no Drive
# Os modelos grandes ficam AQUI no Colab de proposito: baixar do site oficial e mais rapido
# do que ler 6 GB do Drive a cada vez que voce liga. So o que importa guardar vai para o Drive.
if PASTA:
    with open('config.txt', 'w', encoding='utf-8') as f:
        json.dump({
            "path_loras":   [os.path.join(PASTA, 'loras')],
            "path_outputs": os.path.join(PASTA, 'saida'),
        }, f, indent=2)
    print("Fooocus configurado para salvar as fotos no seu Drive.")

# ------------------------------------------------------------------ 5. os dois remendos
# O Fooocus exige a versao 1.26 do numpy. O CuPy que ja vem pronto no Colab foi feito
# para o numpy 2, entao ele quebra e derruba o painel na hora de ligar. O painel nao usa
# esse recurso, entao instalamos as pecas aqui e trocamos os dois arquivos que chamam o
# CuPy por substitutos vazios.
print("\nPreparando as pecas do programa. Leva de 2 a 4 minutos...")
!python -m pip install -q -r requirements_versions.txt

import importlib.util, glob
importlib.invalidate_caches()
_pastas = []
_spec = importlib.util.find_spec('pymatting')
if _spec and _spec.submodule_search_locations:
    _pastas += list(_spec.submodule_search_locations)
for _padrao in ('/usr/local/lib/python3*/dist-packages/pymatting',
                '/usr/local/lib/python3*/site-packages/pymatting',
                '/usr/lib/python3*/dist-packages/pymatting'):
    _pastas += glob.glob(_padrao)

_trocados = 0
for _pasta in dict.fromkeys(_pastas):
    for _nome in ('estimate_foreground_ml_cupy', 'estimate_foreground_ml_pyopencl'):
        _alvo = os.path.join(_pasta, 'foreground', _nome + '.py')
        if os.path.isfile(_alvo):
            with open(_alvo, 'w', encoding='utf-8') as f:
                f.write('def %s(*args, **kwargs):\n    raise RuntimeError(\"desativado no Colab\")\n' % _nome)
            _trocados += 1

print("CuPy neutralizado em %d arquivo(s) - o painel nao precisa dele." % _trocados)
if _trocados == 0:
    print("AVISO: nao achei os arquivos do CuPy. Se o painel nao ligar, avise.")


# --- remendo 2: o Gradio 3.41.2 x o Starlette novo do Colab ---------------------------
# O Gradio pede a montagem da pagina na ordem antiga (nome, dados). O Starlette que vem no
# Colab hoje so aceita a ordem nova (pedido, nome, dados) e responde 'unhashable type: dict'.
# Efeito: o painel liga, mas a pagina abre em branco/erro. Aqui ensinamos o Fooocus a
# tentar de novo na ordem nova quando a antiga falha.
_q = chr(10)
_arq = '/content/Fooocus/modules/ui_gradio_extensions.py'
_velho = _q.join(['    def template_response(*args, **kwargs):',
                  '        res = GradioTemplateResponseOriginal(*args, **kwargs)']) + _q
_novo = _q.join(['    def template_response(*args, **kwargs):',
                 '        try:',
                 '            res = GradioTemplateResponseOriginal(*args, **kwargs)',
                 '        except TypeError:',
                 '            _nome = args[0]',
                 "            _ctx = dict(args[1]) if len(args) > 1 else dict(kwargs.get('context') or {})",
                 "            _req = _ctx.pop('request', None)",
                 '            res = GradioTemplateResponseOriginal(_req, _nome, _ctx)']) + _q
_texto = open(_arq, encoding='utf-8').read()
if 'except TypeError' in _texto:
    print('Gradio: o remendo da pagina ja estava aplicado.')
elif _velho in _texto:
    open(_arq, 'w', encoding='utf-8').write(_texto.replace(_velho, _novo))
    print('Gradio remendado para o Starlette novo do Colab.')
else:
    print('AVISO: nao consegui remendar o Gradio - o arquivo mudou de forma.')

# ------------------------------------------------------------------ 6. ligar
print("\n" + "=" * 72)
print("LIGANDO O PAINEL. Na primeira vez ele baixa o modelo (uns 7 GB),")
print("entao o link pode levar de 5 a 10 minutos para aparecer. E normal.")
print("Quando aparecer, clique no endereco que termina em  .gradio.live")
print("=" * 72 + "\n")

!python launch.py --share --always-high-vram

